# Hospital readmissions: preparing realistic data for a model

The Titanic notebooks show the basics. Real hospital data has problems the Titanic doesn't: **several admissions per patient**, **dates that matter**, **diagnosis codes with a long tail of rare values**, **lab results missing for a reason**, and **columns recorded after the outcome**.

This notebook uses `data/synthetic_admissions.csv`: 4,090 made-up admissions for 1,500 made-up patients, generated by `data/make_admissions.py` from a fixed seed. **No real patient is behind any row.** The task: predict whether an admission is followed by a readmission within 30 days.

In [1]:
import tempfile
import warnings
from pathlib import Path

import joblib
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score

import dstoolkit as dst

raw = pd.read_csv('../data/synthetic_admissions.csv')
print(f"{len(raw):,} admissions, {raw['patient_id'].nunique():,} patients, "
      f"{raw['readmitted_30d'].eq('yes').mean():.1%} readmitted")
raw.head()

4,090 admissions, 1,500 patients, 19.3% readmitted


,admission_id,patient_id,admitted,discharged,age,sex,admission_type,diagnosis_code,prior_admissions_12m,hba1c,readmission_cost,followup_outcome,readmitted_30d
0,A000219,P00080,2023-01-01,2023-01-07,40.0,F,Emergency,I05,3,NaN,NaN,Doing well,no
1,A000284,P00105,2023-01-01,2023-01-05,59.0,F,Emergency,I02,0,7.7,10228.58,No answer,yes
2,A002136,P00776,2023-01-01,2023-01-10,34.0,F,Elective,I09,3,NaN,NaN,No answer,no
3,A002994,P01102,2023-01-01,2023-01-04,24.0,M,Urgent,I07,3,6.8,NaN,No answer,no
4,A003249,P01197,2023-01-01,2023-01-04,74.0,M,Urgent,I02,2,NaN,7066.12,Doing well,yes


## 1. What the profile finds on its own

Naming the target is enough for the report to look for leaks, related features and blanks that carry information. `charts=False` and `related=False` keep the output short.

In [2]:
report = dst.profile(raw, target='readmitted_30d', charts=False, related=False)
for s in report.suggestions:
    print(f"[{s.severity:>6}] {s.column}: {s.message[:150]}")

[  high] readmission_cost: Possible leak: whether readmission_cost is blank predicts readmitted_30d almost perfectly (Cramér's V = 1.00). It may have been recorded after the out
[medium] readmission_cost: 81% missing; too sparse to fill reliably. Drop it, or keep only a 'has value' flag.
[medium] admitted: Dates stored as text; convert to dates so they sort, filter and subtract correctly.
[medium] discharged: Dates stored as text; convert to dates so they sort, filter and subtract correctly.
[medium] readmitted_30d: Yes/No answers stored as text; convert to True/False.
[  high] hba1c: 2,168 missing (53.0%); fill with the median (robust to the skew and outliers).
[medium] age: 127 missing (3.1%); fill with the mean.
[   low] sex: Two categories; label-encode as 0/1 for modeling.
[   low] admission_type: 3 categories; one-hot encode for modeling.
[   low] diagnosis_code: 189 categories; one-hot encode (adds 189 columns). Group rare categories into 'Other' first if many are small (dst.Cat

What it caught without being told:

- **`readmission_cost` is a leak.** It's blank unless the patient was readmitted, so whether it's blank gives the answer away (Cramér's V = 1.00).
- **`patient_id` repeats:** 1,230 patients have more than one admission. A random split would put the same patient in training and test folds.
- **`diagnosis_code` has 189 codes**, most of them rare.
- **Blanks in `hba1c` are related to readmission.** The test is ordered more often for sicker patients, so a missing result is information.
- **`admitted` and `discharged` are dates stored as text.**

What it can't know: `followup_outcome` comes from a phone call 30 days after discharge. Its relation to readmission is strong (0.65), but not strong enough to look like a leak. Only someone who knows how the data was collected can say it was recorded after the outcome.

In [3]:
report.target.features[['column_2', 'measure', 'strength']].round(2)

,column_2,measure,strength
0,followup_outcome,Cramér's V,0.65
1,age,eta,0.22
2,prior_admissions_12m,eta,0.17
3,days from admitted to discharged,eta,0.17
4,admission_type,Cramér's V,0.11
5,hba1c,eta,0.09
6,discharged (as a date),eta,0.02
7,admitted (as a date),eta,0.02
8,sex,Cramér's V,0.00


## 2. Tell it what you know

- `after_outcome=` marks columns recorded after the outcome, so they're treated as leaks.
- `group=` names the patient column, so the script splits by patient: each patient's admissions stay in one fold.

In [4]:
report = dst.profile(raw, target='readmitted_30d', after_outcome=['followup_outcome'], group='patient_id',
                     charts=False, related=False)
report.target.leaks

['followup_outcome', 'readmission_cost']

## 3. A leak-free script

`pipeline()` writes the modelling script. The `hba1c` flag is safe to add, because whether the test was ordered is known during the stay, before the outcome: `missing_flags=['hba1c']` adds it.

In [5]:
script = report.pipeline(missing_flags=['hba1c'])
print(script)

import dstoolkit as dst
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline

# 1. Steps that learn nothing from the data: safe before the split
df = dst.to_datetime_columns(df, ['admitted', 'discharged'])
# Was-missing flags you chose (they learn nothing from the data)
df['hba1c_missing'] = df['hba1c'].isna().astype(int)

# 2. Features and target
# Left out (recorded after the outcome (after_outcome)): followup_outcome
# Left out (possible leaks: check they were known before the outcome): readmission_cost
# Left out (the group column: used to split, not to predict): patient_id
# Left out (identifiers or free text): admission_id
X = df[['admitted', 'discharged', 'age', 'sex', 'admission_type', 'diagnosis_code', 'prior_admissions_12m', 'hba1c', 'hba1c_missing']]
y = df['readmitted_30d']
groups = df['patient_id']   # rows of

Read before running:

- The dates become features through `DateFeatures`, inside the pipeline: month and day of week, but not year, because new admissions will come from years the model never saw. The script's comment points out that a **duration** often matters more than the dates themselves. Here the length of stay drives readmission, so the next cell adds it with `intervals=`.
- `diagnosis_code` is encoded with codes under 1% of rows grouped into "Other", which turns 189 codes into about a dozen columns.
- The folds are `StratifiedGroupKFold`: grouped by patient, keeping the share of readmissions in each fold.

The script runs in its own namespace (`run`), and the next cells take the model, data and folds from it.

In [6]:
script = script.replace(
    "parts=['month', 'dayofweek']",
    "parts=['month', 'dayofweek'], intervals={'stay_days': ('admitted', 'discharged')}")
run = {'df': raw.copy()}
with warnings.catch_warnings():
    warnings.simplefilter('ignore')   # pandas notices from the date conversion, not the model
    exec(script, run)
model, X, y, folds, groups, scores = (run[name] for name in ['model', 'X', 'y', 'folds', 'groups', 'scores'])

Converted to dates: admitted, discharged.


model:    roc_auc 0.750 ± 0.031
baseline: roc_auc 0.500   (no skill: the score to beat)


The model scores well above the no-skill baseline of 0.5. The data was generated with plenty of randomness in who gets readmitted, so no model could score near 1 honestly.

## 4. What the grouped folds protect against

The same model, scored with ordinary random folds, so a patient's admissions can be in training and test data at once:

In [7]:
random_folds = StratifiedKFold(5, shuffle=True, random_state=0)
random_scores = cross_val_score(model, X, y, cv=random_folds, scoring='roc_auc')
print(f"random folds:  roc_auc {random_scores.mean():.3f}")
print(f"grouped folds: roc_auc {scores.mean():.3f}   (the honest estimate for new patients)")

random folds:  roc_auc 0.760
grouped folds: roc_auc 0.750   (the honest estimate for new patients)


Here the gap is small, because few of the features describe the patient rather than the admission. It grows with features that identify a patient (date of birth, address, a long medical history): the model then recognises patients it has already seen, which it can't do for new ones. Grouped folds give the estimate that holds for new patients, whatever the data.

### A time split

A model built today will be used on next month's admissions. `pipeline(time_column='admitted')` sorts the admissions by date and trains each fold on earlier admissions, testing on later ones:

In [8]:
time_script = report.pipeline(missing_flags=['hba1c'], time_column='admitted').replace(
    "parts=['month', 'dayofweek']",
    "parts=['month', 'dayofweek'], intervals={'stay_days': ('admitted', 'discharged')}")
timed = {'df': raw.copy()}
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    exec(time_script, timed)

Converted to dates: admitted, discharged.


model:    roc_auc 0.754 ± 0.021
baseline: roc_auc 0.500   (no skill: the score to beat)


This is the closest of the three to how the model will actually be used. A returning patient can now be in training and test data, which is realistic: the model will see returning patients in practice too. If the score here were much lower than with grouped folds, the patterns would be shifting over time, and the model would need retraining more often.

## 5. Can the risks be trusted?

ROC AUC only checks the ranking. A care team acting on the model also needs the risks themselves to be right: of the patients given 30%, about 30% should be readmitted. And it needs to know what a realistic intervention catches, such as calling the riskiest 10% of patients.

Each admission gets a risk from a model that never saw that patient (out-of-fold predictions, using the same grouped folds). The admissions are then split into ten equal bands, from lowest to highest risk:

In [9]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import cross_val_predict

readmitted = y.eq('yes')
yes = sorted(y.unique()).index('yes')


def out_of_fold_risk(estimator):
    risk = cross_val_predict(estimator, X, y, cv=folds, groups=groups, method='predict_proba')[:, yes]
    return pd.Series(risk, index=X.index)


def calibration(risk):
    bands = pd.qcut(risk, 10, labels=range(1, 11))
    return pd.DataFrame({'predicted': risk, 'actual': readmitted}).groupby(bands, observed=True).mean().round(3)


risk = out_of_fold_risk(model)
calibration(risk)

,predicted,actual
1,0.036,0.034
2,0.062,0.046
3,0.082,0.078
4,0.105,0.115
5,0.133,0.144
6,0.166,0.183
7,0.206,0.203
8,0.264,0.264
9,0.342,0.335
10,0.519,0.526


In each band, `predicted` is the average risk the model gave and `actual` is the share actually readmitted. The closer the two columns, the more the risks can be taken at face value.

Two single numbers summarise it. The **Brier score** is the average squared gap between each risk and what happened: lower is better, and always guessing the overall rate sets the bar to beat. The **riskiest 10%** shows what a targeted follow-up programme would catch:

In [10]:
base_rate = readmitted.mean()
print(f"Brier score: {brier_score_loss(readmitted, risk):.3f} "
      f"(always guessing {base_rate:.0%}: {brier_score_loss(readmitted, [base_rate] * len(y)):.3f})")
top = risk >= risk.quantile(0.9)
print(f"riskiest 10%: {readmitted[top].mean():.0%} readmitted, against {base_rate:.0%} of all admissions; "
      f"they include {readmitted[top].sum() / readmitted.sum():.0%} of all readmissions")

Brier score: 0.135 (always guessing 19%: 0.156)
riskiest 10%: 53% readmitted, against 19% of all admissions; they include 27% of all readmissions


**Why the script's model is shallow.** `pipeline()` uses gradient boosting with small, slowly learning trees (`max_depth=3`, `learning_rate=0.03`, `min_samples_leaf=40`). With scikit-learn's default settings, the same model fits this data too closely, and its risks drift from what actually happens:

In [11]:
loose = clone(model).set_params(histgradientboostingclassifier=HistGradientBoostingClassifier(random_state=0))
loose_risk = out_of_fold_risk(loose)
print(f"default settings: Brier {brier_score_loss(readmitted, loose_risk):.3f}")
calibration(loose_risk).loc[[1, 10]]

default settings: Brier 0.145


,predicted,actual
1,0.010,0.046
10,0.626,0.479


With the default settings, the riskiest band is given a much higher risk than the share actually readmitted, and the safest band a much lower one. A care team would chase false alarms at the top and miss patients at the bottom. The gentler settings are why the script's risks line up.

## 6. What the leak would have done

Adding back whether `readmission_cost` is blank:

In [12]:
leaky_X = X.assign(has_readmission_cost=raw['readmission_cost'].notna().astype(int))
leaky = cross_val_score(model, leaky_X, y, cv=folds, scoring='roc_auc', groups=groups)
print(f"with the leak: roc_auc {leaky.mean():.3f}")

with the leak: roc_auc 1.000


A perfect score, from a column that only exists once the answer is known. The model would be worthless on the day it's needed.

## 7. Save the fitted model and use it later

Fit on all the data, save with `joblib`, load it back and check the predictions match:

In [13]:
model.fit(X, y)
path = Path(tempfile.mkdtemp()) / 'readmission_model.joblib'
joblib.dump(model, path)
reloaded = joblib.load(path)
same = (reloaded.predict_proba(X.head(100)) == model.predict_proba(X.head(100))).all()
print(f"saved {path.stat().st_size / 1024:.0f} KB; predictions match after reloading: {same}")

saved 208 KB; predictions match after reloading: True


## 8. Is new data still like the training data?

A model is only as good as the match between the data it learned from and the data it scores. Before scoring a new batch, `dst.compare()` checks every column: did the type change, did the distribution shift (the population stability index, PSI), are there more blanks, are there values never seen before?

First, the 2024 admissions against the 2023 ones. Same hospital, same process, so nothing should stand out:

In [14]:
pd.set_option('display.max_colwidth', 90)
earlier = raw[raw['admitted'] < '2024-01-01']
later = raw[raw['admitted'] >= '2024-01-01']
dst.compare(earlier, later)[['column', 'status', 'kind', 'shift_psi', 'issues']].round(3)

Compared 13 column(s): 13 stable.


,column,status,kind,shift_psi,issues
0,readmission_cost,stable,number,0.031,
1,hba1c,stable,number,0.024,
2,age,stable,number,0.021,
3,diagnosis_code,stable,category,0.007,"7 new value(s), 1.0% of new rows: R125, R047, R131, R014, R058, ..."
4,admission_type,stable,category,0.006,
5,prior_admissions_12m,stable,number,0.005,
6,readmitted_30d,stable,category,0.001,
7,followup_outcome,stable,category,0.001,
8,sex,stable,category,0.000,
9,admission_id,stable,identifier or free text,NaN,identifier or free text: distribution not compared


All stable. `diagnosis_code` lists a few rare codes seen for the first time, about 1% of rows: normal for a long tail of codes, so they're listed without raising the status.

Now a batch with the kind of problems that break models without any error message. In this made-up batch:

- the lab switched HbA1c from percent to mmol/mol
- the admission system started writing "ER" instead of "Emergency"
- a new diagnosis code appeared
- an interface change dropped prior-admission counts for some patients

A model would score all of it, and the risks would quietly be wrong.

In [15]:
batch = later.copy()
batch['hba1c'] = (batch['hba1c'] * 10.929 - 23.5).round()                      # percent -> mmol/mol
batch['admission_type'] = batch['admission_type'].replace({'Emergency': 'ER'})
batch.loc[batch.sample(frac=0.08, random_state=0).index, 'diagnosis_code'] = 'U07'
batch.loc[batch.sample(frac=0.15, random_state=1).index, 'prior_admissions_12m'] = None

changes = dst.compare(earlier, batch)
changes[changes['status'] != 'stable'][['column', 'status', 'shift_psi', 'blanks_after', 'issues']].round(3)

Compared 13 column(s): 4 major; 9 stable.


,column,status,shift_psi,blanks_after,issues
0,admission_type,major,8.492,0.000,"distribution shifted (PSI 8.49); 1 new value(s), 53.2% of new rows: ER"
1,hba1c,major,6.597,0.523,distribution shifted (PSI 6.60)
2,diagnosis_code,major,0.027,0.000,"7 new value(s), 8.9% of new rows: U07, R125, R131, R047, R014, ..."
3,prior_admissions_12m,major,0.007,0.150,blanks 0.0% → 15.0%


Every problem is caught and explained. A check like this belongs at the start of every scoring run: stop, or at least warn, when a column comes back "major". In a cloud setup it's the kind of check an upload-triggered function can run before the data reaches the model.

Dates are expected to move forward, so their distribution isn't compared, and identifiers like `patient_id` only get the type and blank checks. A p-value is reported too, but doesn't set the status: with thousands of rows, tiny harmless shifts are "significant".

## Summary

| Problem in the data | What dstoolkit did |
|---|---|
| Several admissions per patient | Warned, and `group=` split the folds by patient |
| A column blank unless readmitted | Flagged it as a leak by itself |
| A follow-up call after discharge | Left out once declared with `after_outcome=` |
| Lab tests missing for a reason | Offered a flag; `missing_flags=` added it |
| 189 diagnosis codes, most rare | Grouped codes under 1% into "Other" |
| Dates stored as text | Converted before the split, turned into features (and a length of stay) in the pipeline |
| A model that will predict future admissions | `time_column=` trained on earlier admissions and tested on later ones |
| Risks a care team will act on | Checked calibration by risk band, the Brier score, and what calling the riskiest 10% would catch |
| New batches with a unit change, a relabelled category, a new code or missing values | `dst.compare()` flagged each one before scoring |

The judgement calls stayed with the analyst: which columns were recorded after the outcome, which blanks were known in time, and whether a duration matters.